In [20]:
import os
import json
import ollama
import json 
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from my_easy_scraper import fetch_website_links, fetch_website_contents, fetch_all_website_links
from openai import OpenAI

In [ ]:
MODEL_OLLAMA = "minimax-m2.5:cloud"
openai_ollama = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama') #didn't end up using this. Used ollama.chat instead


load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')
MODEL_GPT = 'gpt-5-nano'
openai_gpt = OpenAI()

website = "https://github.com"

In [26]:

"""
Since I am using a small model for this, I am using similar system and user prompts for reinforcing.
"""

link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_all_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt


In [27]:
def select_relevant_links(url):
    response = openai_gpt.chat.completions.create(
        model=MODEL_GPT,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links


In [28]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [29]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt
    return user_prompt



In [ ]:
# def create_brochure(company_name, url):
#     response = openai.chat.completions.create(
#         model="gpt-4.1-mini",
#         messages=[
#             {"role": "system", "content": brochure_system_prompt},
#             {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
#         ],
#     )
#     result = response.choices[0].message.content
#     display(Markdown(result))


"""
Using Streaming and OLLAMA here to decrease TTFT and since I am not truncating any content from the websites so the GPT API might become too expensive
"""

def stream_brochure(company_name, url):
    stream = ollama.chat(
        model=MODEL_OLLAMA,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk["message"]["content"] 
        update_display(Markdown(response), display_id=display_handle.display_id)

In [37]:
stream_brochure("Github",website)

Crawling pages: 100%|██████████| 1/1 [00:00<00:00,  1.48 pages/s]


# GitHub Company Brochure

## The World's Leading AI-Powered Developer Platform

GitHub is the complete developer platform to build, scale, and deliver secure software. Trusted by millions of developers and 90% of the Fortune 100, GitHub brings together the tools, AI, and teams needed to accelerate the entire software development lifecycle—from code to deployment.

---

## Platform Overview

GitHub provides a unified, AI-powered platform designed for the modern developer workflow:

*   **AI Code Creation**: 
    *   **GitHub Copilot**: Your AI pair programmer that writes code faster, suggests entire functions, and helps explain unfamiliar code.
    *   **GitHub Models**: Manage and compare prompts for AI workflows.
*   **Developer Workflows**:
    *   **GitHub Actions**: Automate any workflow with CI/CD.
    *   **GitHub Codespaces**: Instant, cloud-based development environments.
    *   **Issues & Projects**: Plan and track work seamlessly alongside your code.
    *   **Code Review**: Manage code changes with collaborative review tools.
*   **Application Security**:
    *   **GitHub Advanced Security**: Find and fix vulnerabilities natively in the workflow.
    *   **Secret Protection**: Stop leaks before they happen with push protection.
    *   **Dependabot**: Automated dependency updates.

---

## Impact by the Numbers

*   **180M+** Developers
*   **4M+** Organizations
*   **420M+** Repositories
*   **90%** of the Fortune 100

---

## Customer Success Stories

GitHub is used by the world’s most innovative companies to solve complex challenges.

*   **Mercedes-Benz**: Unified 55,000+ developers on GitHub, accepting over 2 million lines of AI-powered code with Copilot. *“GitHub is a complete platform that frees us from menial tasks and enables us to do our best work.”* — Fabian Faulhaber, Application Manager, Mercedes-Benz
*   **Duolingo**: Achieved a **25% increase in developer speed** and a **67% decrease in code review turnaround time** using GitHub Copilot and Codespaces.
*   **Mercado Libre**: The largest e-commerce in Latin America merges **100,000 pull requests per day**, with developers coding **~50% faster** thanks to GitHub Copilot.
*   **Figma**: Streamlined development and strengthened security by adopting GitHub Enterprise and innersource practices.

---

## Company Culture & Social Impact

At GitHub, we believe software development should be accessible, inclusive, and sustainable.

### Diversity, Inclusion, & Belonging
We are dedicated to building a team that reflects the global developer community. We invest in:
*   **Global Accessibility**: Localization efforts and outreach (e.g., GitHub for Mobile available in 5 languages).
*   **Learning & Development**: $2,000 USD annual learning budget for every employee.
*   **Employee Benefits**: Access to mental health services, wellness stipends, and back-up care support.
*   **Training**: Mandatory unconscious bias, privilege, and allyship training for all employees.

### Social Impact
GitHub empowers positive change through:
*   **GitHub for Nonprofits**: Verified nonprofits get exclusive access to free or discounted plans.
*   **GitHub Gives**: Employees receive matching donations and paid time off to volunteer.
*   **Policy Advocacy**: Fighting for developer rights and inclusive opportunities.

---

## Join the Team

GitHub is a passionate group of people dedicated to building the home for all developers. We are constantly looking for talented individuals to help us shape the future of software.

Explore open positions and learn more about our culture on our [Careers page](#).

---

## Plans & Pricing

GitHub offers flexible plans for teams of any size:

*   **Free**: The basics for individuals and organizations ($0/month).
*   **Team**: Advanced collaboration ($4/user/month).
*   **Enterprise**: Security, compliance, and flexible deployment (Starting at $21/user/month).

**Add-ons** include GitHub Copilot, Advanced Security, and Premium Support.

---

## Get Started

Whether you are a startup or a Fortune 500 enterprise, GitHub helps you innovate securely on the platform developers love.

*   [Start a free trial](#)
*   [Contact Sales](#)
*   [View Customer Stories](#)